# NeurIPS — cross-method comparison

Loads every per-method pickle written by the per-method evaluation notebooks
(`neurips_diff_eval.ipynb` etc.) and renders multi-method tables + plots in
the unified paper style. Adding a new method = drop another pickle into
`RESULTS_DIR` and re-run this notebook end-to-end.

In [ ]:
%load_ext autoreload
%autoreload 2

import sys, os
sys.path.append("..")
sys.path.append(os.path.dirname(os.path.abspath('.')))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from neurips_paper_plots import (
    apply_paper_style, save_fig, load_method_results,
    pretty_scatter, pretty_grouped_bars,
    METHOD_PALETTE, COLOR_REF, COLOR_GYROFLOW, COLOR_GYROSWIN, COLOR_GKW,
    darken, lighten,
)

apply_paper_style()
RESULTS_DIR = "/system/user/galletti/git/neural-gyrokinetics-gitlab/notebooks/figs/paper"
COMPARE_DIR = os.path.join(RESULTS_DIR, "compare")
os.makedirs(COMPARE_DIR, exist_ok=True)


## 1. Load all per-method pickles

In [ ]:
results = load_method_results(RESULTS_DIR)
print(f"loaded {len(results)} methods: {sorted(results.keys())}")
for name, r in results.items():
    print(f"  {name:>14s}  ID={len(r.get('ID_trajs', []))}  OOD={len(r.get('OOD_trajs', []))}  "
            f"keys={sorted(r.keys())}")


## 2. FID summary across methods

One bar group per U-Net level (skip_deep / bottleneck / flux_head / phi);
one bar per method. Mean over all available trajectories per method.

In [ ]:
rows = []
for name, r in results.items():
    fids = r.get("fids", {})
    for lv, traj_map in fids.items():
        vals = [v for v in traj_map.values() if np.isfinite(v)]
        if vals:
            rows.append({"method": name, "level": lv, "mean_fid": float(np.mean(vals)),
                         "n": len(vals)})
fid_long = pd.DataFrame(rows)
if fid_long.empty:
    print("no FID data found across methods")
else:
    fid_pivot = fid_long.pivot_table(index="level", columns="method", values="mean_fid")
    print(fid_pivot.to_string(float_format=lambda x: f"{x:.4f}"))
    fig, ax = plt.subplots(figsize=(1.6 * max(len(fid_pivot.index), 2), 3.2))
    pretty_grouped_bars(ax, fid_pivot, palette=METHOD_PALETTE)
    ax.set_ylabel("mean FID")
    ax.set_title("FID across U-Net levels — methods compared")
    fig.tight_layout()
    save_fig(fig, "compare_fid", COMPARE_DIR)
    plt.show()


## 3. Distributional metrics across methods

For each (family ∈ {flux, ky_spec, fluxspec}, base ∈ {mmd, ad, ks_p, w1,
arima_l2, r2}) we tabulate the per-method mean.

In [ ]:
SUMMARY_MEASURES = ["mmd", "ad", "ks_p", "w1", "arima_l2", "r2"]
FAMILIES = ["flux", "ky_spec", "fluxspec"]

rows = []
for name, r in results.items():
    metrics = r.get("metrics", {})
    for fam in FAMILIES:
        for m in SUMMARY_MEASURES:
            col = f"{fam}_{m}"
            if col in metrics:
                vals = [v for v in metrics[col].values() if isinstance(v, (int, float)) and np.isfinite(v)]
                if vals:
                    rows.append({"method": name, "family": fam, "measure": m,
                                  "mean": float(np.mean(vals)), "n": len(vals)})
div_long = pd.DataFrame(rows)
if div_long.empty:
    print("no distributional metrics found")
else:
    for fam in FAMILIES:
        sub = div_long[div_long["family"] == fam]
        if sub.empty:
            continue
        pv = sub.pivot_table(index="measure", columns="method", values="mean")
        print(f"\n=== {fam} ===")
        print(pv.to_string(float_format=lambda x: f"{x:.4g}"))
        fig, ax = plt.subplots(figsize=(1.5 * max(len(pv.columns), 2), 3.2))
        pretty_grouped_bars(ax, pv, palette=METHOD_PALETTE)
        ax.set_ylabel(f"{fam} — mean")
        ax.set_title(f"{fam} — distributional metrics across methods")
        fig.tight_layout()
        save_fig(fig, f"compare_div_{fam}", COMPARE_DIR)
        plt.show()


## 4. Probe scatter — one panel per method

In [ ]:
methods_with_probe = [m for m, r in results.items() if r.get("probe", {}).get("y_true") is not None]
if not methods_with_probe:
    print("no probe data")
else:
    fig, axes = plt.subplots(1, len(methods_with_probe),
                              figsize=(3.0 * len(methods_with_probe), 3.2),
                              sharex=True, sharey=True, squeeze=False)
    for ax, name in zip(axes.flat, methods_with_probe):
        p = results[name]["probe"]
        col = METHOD_PALETTE.get(name, "#666")
        pretty_scatter(ax, np.asarray(p["y_true"]), np.asarray(p["y_pred_gen"]),
                        color=col, label=name, diag=True,
                        annot=f"RMSE={p.get('rmse', float('nan')):.3f}\nr={p.get('pearson', float('nan')):.3f}")
        ax.set_title(name)
        ax.set_xlabel("ground-truth Q")
    axes[0, 0].set_ylabel("predicted Q")
    fig.suptitle("Flux probe — generative latents (per method)", y=1.02, fontsize=10)
    fig.tight_layout()
    save_fig(fig, "compare_probe_flux", COMPARE_DIR)
    plt.show()


## 5. Qualitative warm-restart overlays

For each ID trajectory present in *all* methods, overlay the warm flux
$Q(t)$ from each method on the same axis with the GT trajectory.

In [ ]:
methods_with_qual = [m for m, r in results.items() if r.get("qualitative", {}).get("warm_eflux")]
if len(methods_with_qual) == 0:
    print("no qualitative data")
else:
    common_trajs = set.intersection(
        *[set(results[m]["qualitative"]["warm_eflux"].keys()) for m in methods_with_qual]
    )
    common_trajs = sorted(common_trajs)
    print(f"{len(common_trajs)} trajectories common to all methods: {common_trajs}")
    for traj in common_trajs[:3]:
        fig, ax = plt.subplots(figsize=(6.5, 3.0))
        any_method = methods_with_qual[0]
        gt_full = results[any_method]["qualitative"]["gt_full_flux"].get(traj)
        if gt_full is not None:
            ax.plot(np.arange(len(gt_full)), gt_full, color=COLOR_REF, lw=0.9,
                     alpha=0.7, label="GKW from t=0")
        L = len(gt_full) if gt_full is not None else 0
        for m in methods_with_qual:
            wf = results[m]["qualitative"]["warm_eflux"].get(traj)
            if wf is None: continue
            wf = np.asarray(wf)
            if L:
                x = np.arange(L - len(wf), L)
            else:
                x = np.arange(len(wf))
            col = METHOD_PALETTE.get(m, None)
            ax.plot(x, wf, lw=1.4, color=col, label=m)
        ax.set_xlabel("step"); ax.set_ylabel(r"$Q(t)$")
        ax.set_title(f"Warm-restart flux overlay — {traj}", fontsize=10)
        ax.legend(fontsize=8)
        fig.tight_layout()
        save_fig(fig, f"compare_warmflux_{traj}", COMPARE_DIR)
        plt.show()


## 6. Combined comparison artefacts

In [ ]:
combined = {
    "fid_long": fid_long.to_dict(orient="records") if 'fid_long' in dir() and not fid_long.empty else [],
    "div_long": div_long.to_dict(orient="records") if 'div_long' in dir() and not div_long.empty else [],
    "methods":  sorted(results.keys()),
}
import pickle
with open(os.path.join(COMPARE_DIR, "combined.pkl"), "wb") as f:
    pickle.dump(combined, f)
print(f"wrote {os.path.join(COMPARE_DIR, 'combined.pkl')}")
